In [1]:
!pip install -q gpt4all sentence-transformers faiss-cpu numpy PyMuPDF
!pip install gradio

In [2]:
import fitz  # PyMuPDF for PDF text extraction
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from gpt4all import GPT4All
import os

In [3]:
models = GPT4All.list_models()
for m in models:
    print(m["filename"])

qwen2.5-coder-7b-instruct-q4_0.gguf
Meta-Llama-3-8B-Instruct.Q4_0.gguf
DeepSeek-R1-Distill-Qwen-7B-Q4_0.gguf
DeepSeek-R1-Distill-Qwen-14B-Q4_0.gguf
DeepSeek-R1-Distill-Llama-8B-Q4_0.gguf
DeepSeek-R1-Distill-Qwen-1.5B-Q4_0.gguf
Llama-3.2-3B-Instruct-Q4_0.gguf
Llama-3.2-1B-Instruct-Q4_0.gguf
Nous-Hermes-2-Mistral-7B-DPO.Q4_0.gguf
mistral-7b-instruct-v0.1.Q4_0.gguf
Meta-Llama-3.1-8B-Instruct-128k-Q4_0.gguf
mistral-7b-openorca.gguf2.Q4_0.gguf
gpt4all-falcon-newbpe-q4_0.gguf
orca-2-7b.Q4_0.gguf
orca-2-13b.Q4_0.gguf
wizardlm-13b-v1.2.Q4_0.gguf
ghost-7b-v0.9.1-Q4_0.gguf
nous-hermes-llama2-13b.Q4_0.gguf
gpt4all-13b-snoozy-q4_0.gguf
mpt-7b-chat-newbpe-q4_0.gguf
mpt-7b-chat.gguf4.Q4_0.gguf
Phi-3-mini-4k-instruct.Q4_0.gguf
orca-mini-3b-gguf2-q4_0.gguf
replit-code-v1_5-3b-newbpe-q4_0.gguf
starcoder-newbpe-q4_0.gguf
rift-coder-v0-7b-q4_0.gguf
all-MiniLM-L6-v2-f16.gguf
all-MiniLM-L6-v2.gguf2.f16.gguf
em_german_mistral_v01.Q4_0.gguf
nomic-embed-text-v1.f16.gguf
nomic-embed-text-v1.5.f16.gguf
qwen2-1_

In [4]:
# Load sentence transformer model
embedder = SentenceTransformer("all-MiniLM-L6-v2")
# mistral -- too long time take output
# Load the GPT4All model --  mistral-7b-instruct-v0.1.Q4_0.gguf, Phi-3-mini-4k-instruct.Q4_0.gguf
model = GPT4All("mistral-7b-instruct-v0.1.Q4_0.gguf")  # Adjust with your correct model path,n_threads=4

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [5]:
def extract_pdf_text(pdf_path):
    """Extract text from a PDF file"""
    document = fitz.open(pdf_path)
    pdf_text = ""

    # Loop through all pages and extract text
    for page_num in range(len(document)):
        page = document.load_page(page_num)
        pdf_text += page.get_text("text")  # Extract text as plain text

    return pdf_text

In [6]:
def build_chunks(text, chunk_size=200, overlap=40):
    words = text.split()

    return [
        " ".join(words[i:i + chunk_size])
        for i in range(0, len(words), chunk_size - overlap)
    ]

In [7]:
def rag_query(query, top_k=2):
    global index, documents
    if index is None or documents is None or len(documents) == 0:
        return "⚠️ Please upload a PDF first !"

    # Embed the query
    query_vec = embedder.encode([query])

    # Retrieve the top-k most similar documents
    distances, indices = index.search(np.array(query_vec), top_k)
    retrieved_docs = [documents[i] for i in indices[0]]

    # Create context
    context = "\n".join(retrieved_docs)


    # Create the prompt for the GPT4All model
    prompt = f"""
You are a helpful assistant. Answer the question using ONLY the provided context.

If the answer is not explicitly present in the context, respond exactly with:
"not found in pdf file"

Context:
{context}

Question:
{query}

Answer:
"""


    # Generate the response using GPT4All model
    # with model.chat_session():
    #     response = model.generate(prompt, max_tokens=100, temp=0.1)

    # with model.chat_session():
    #   response = model.generate(
    #       prompt,
    #       max_tokens=20,
    #       temp=0.1,
    #       top_k=20, #top 20
    #       top_p=0.8, # probility
    #       repeat_penalty=1.1, # Penalizes repeating words/phrases

    #   )

    # return response

    response = model.generate(
        prompt,
        max_tokens=50,
        #temp=0.1,

    )

    return response.strip().split("\n")[0]

In [8]:
def load_pdf(pdf_file):
    global documents, index

    # check if file is uploaded
    if pdf_file is None:
        return "⚠️ Please upload a PDF first before clicking Load"

    file_path = pdf_file.name
    ext = os.path.splitext(file_path)[1].lower()

    # ---------- PDF ----------
    if ext == ".pdf":
        text = extract_pdf_text(file_path)

    # ---------- TXT ----------
    elif ext == ".txt":
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read()

    else:
        return "⚠️ Unsupported file type"


    text = extract_pdf_text(pdf_file)
    documents = build_chunks(text)

    embeddings = embedder.encode(documents)

    import faiss
    dim = embeddings.shape[1]

    index = faiss.IndexFlatL2(dim)
    index.add(np.array(embeddings))
    return f"✅ File loaded successfully ({ext}, {len(documents)} chunks)"

In [10]:
import gradio as gr

# Sample questions
question_list = [
    "What is the purpose of FAISS?",
    "Where is HCL headquartered?",
    "How many employees does HCL have?",
    "What services does HCL provide?",
    "How does HCL promote innovation?"
]


# RAG wrapper
def ask_question(q):
    return rag_query(q)


# ---------- UI ----------
with gr.Blocks() as app:

    gr.Markdown("# 📄 PDF RAG  with GPT4All Chatbot")

    # PDF upload
    file_input = gr.File(label="Upload PDF or TXT", file_types=[".pdf", ".txt"])
    load_btn = gr.Button("Load File")
    status = gr.Textbox(label="Status")

    load_btn.click(load_pdf, inputs=file_input, outputs=status)

    # ---------- Dropdown ----------
    gr.Markdown("## ❓ Select a Question")

    question_dropdown = gr.Dropdown(
        choices=question_list,
        label="Choose a question",
        value=question_list[0]  # default selected
    )

    ask_btn = gr.Button("Get Answer")
    answer = gr.Textbox(label="Answer")

    ask_btn.click(ask_question, inputs=question_dropdown, outputs=answer)

    # ---------- Free text ----------
    gr.Markdown("## 💬 Or Ask Your Own Question")

    custom_q = gr.Textbox(label="Type your question")
    custom_btn = gr.Button("Ask")

    custom_btn.click(ask_question, inputs=custom_q, outputs=answer)

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://2b538d5b2597926f23.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
